# Energy & Utilities Document Q&A (RAG)
Retrieval-Augmented Generation app that answers questions about US energy and utilities reports (NERC, FERC, EIA) and cites the source file and page for each answer.

**Stack:** LangChain, Chroma, sentence-transformers (all-MiniLM-L6-v2), Claude API

## 1. Install dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface langchain-anthropic sentence-transformers chromadb pypdf

## 2. Load API key
The key is read from a Colab secret named `ANTHROPIC_API_KEY` (never hard-coded in the notebook).

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("Key loaded")

## 3. Upload PDF reports

In [ ]:
import os
from google.colab import files
os.makedirs("docs", exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"docs/{name}", "wb") as f:
        f.write(data)
print(os.listdir("docs"))

## 4. Ingest: load, chunk, embed, and index
PDFs are loaded page by page, split into 1,000-character chunks with 150-character overlap, embedded locally, and stored in a Chroma vector database.

In [ ]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # free, runs locally

pages = []
for pdf in sorted(Path("docs").glob("*.pdf")):
    loaded = PyPDFLoader(str(pdf)).load()
    print(f"Loaded {pdf.name}: {len(loaded)} pages")
    pages.extend(loaded)

chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(pages)
print(f"Split into {len(chunks)} chunks")

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
db = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")
print("Vector database ready")

## 5. Retrieval and generation chain
The 4 most similar chunks are retrieved and passed to Claude with a prompt that restricts answers to the retrieved context.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

retriever = db.as_retriever(search_kwargs={"k": 4})
llm = ChatAnthropic(model="claude-sonnet-5-5")
prompt = ChatPromptTemplate.from_template(
    """You are an assistant answering questions about energy and utilities documents.
Use ONLY the context below. If the answer is not in the context, say you could not find it.

Context:
{context}

Question: {question}

Answer concisely."""
)
chain = prompt | llm

def _text(content):
    """Claude may return a list of blocks (reasoning + text); keep only the text."""
    if isinstance(content, str):
        return content
    return "".join(b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text")

def ask(question):
    docs = retriever.invoke(question)  # RETRIEVAL: find relevant chunks
    context = "\n\n".join(d.page_content for d in docs)
    reply = chain.invoke({"context": context, "question": question})  # GENERATION
    sources = sorted({f"{os.path.basename(d.metadata.get('source','?'))} (p. {d.metadata.get('page',0)+1})" for d in docs})
    print(_text(reply.content))
    print("\nSources:", ", ".join(sources))

## 6. Example queries

In [ ]:
ask("What are the main risks or challenges discussed in these documents?")

In [ ]:
ask("How is data center growth expected to affect electricity demand?")

In [ ]:
ask("What did FERC say about natural gas price volatility?")